In [58]:
import pandas as pd

df_marking = pd.read_csv('df_with_marking_final.csv')

In [3]:
labels = [
    "GENDER_ACCUSED", "ALCOHOL", "DRUGS", "MENTAL_DISORDER", "PRIOR_CONVICTIONS",
    "GENDER_VICTIM", "RELATIONSHIP", "TIME_OF_DAY", "DAY_OF_WEEK", "SEASON", "YEAR",
    "CRIME_REGION", "LOCATION", "METHOD", "CRUELTY", "INTENTIONAL", "MOTIVE",
    "WITNESSES", "PRECRIME_ARGUMENT", "PRISON_TERM"
]

In [ ]:
male_keywords = ["подсудимый", "обвиняемый", "осуждённый", "обвиняемого", "подсудимого", "Подсудимый"]
female_keywords = ["подсудимая", "обвиняемая", "осуждённая", "подсудимой"]

def get_full_text(row):
    return f"{str(row['preamble'])} {str(row['description'])} {str(row['sentence'])}"

In [53]:
train_data = []

for idx, row in df_marking.iterrows():
    text = get_full_text(row)
    text_lower = text.lower()
    gender = row["gender_accused"]

    if gender == "неизвестно" or pd.isnull(gender):
        continue

    keywords = female_keywords if gender == "женщина" else male_keywords
    found = False

    for keyword in keywords:
        start = text_lower.find(keyword)
        if start != -1:
            end = start + len(keyword)
            train_data.append((text, {"entities": [(start, end, "GENDER_ACC")], "gender": gender}))  # <== ДОБАВИЛИ
            found = True
            break

    if not found:
        print(f"Не удалось найти ключевое слово для пола '{gender}' в тексте id={row['id']}")

print(f"TRAIN_DATA готово: {len(train_data)} примеров")

TRAIN_DATA готово: 100 примеров


In [54]:
train_examples, valid_examples = train_test_split(train_data, test_size=0.2, random_state=42)

In [5]:
import spacy
from spacy.training.example import Example
from spacy.util import minibatch
import random

nlp = spacy.blank("ru")

ner = nlp.add_pipe("ner")

ner.add_label("GENDER_ACC")

examples = []
for text, annotations in train_data:
    doc = nlp.make_doc(text)
    examples.append(Example.from_dict(doc, annotations))

n_iter = 30
optimizer = nlp.begin_training()

for i in range(n_iter):
    random.shuffle(examples)
    losses = {}
    batches = minibatch(examples, size=4)

    for batch in batches:
        nlp.update(batch, losses=losses)

    print(f"Итерация {i+1}/{n_iter} — потери: {losses}")

nlp.to_disk("ner_gender_model")
print("Модель сохранена в папке 'ner_gender_model'")

/Users/ekaterinastepura/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


Итерация 1/30 — потери: {'ner': 127679.25940419802}
Итерация 2/30 — потери: {'ner': 1407.507149199042}
Итерация 3/30 — потери: {'ner': 121.38215350258253}
Итерация 4/30 — потери: {'ner': 91.00199238920834}
Итерация 5/30 — потери: {'ner': 67.73362063160044}
Итерация 6/30 — потери: {'ner': 51.1388525337536}
Итерация 7/30 — потери: {'ner': 46.11303206540158}
Итерация 8/30 — потери: {'ner': 38.90499051983647}
Итерация 9/30 — потери: {'ner': 37.647491415914104}
Итерация 10/30 — потери: {'ner': 27.8497659977967}
Итерация 11/30 — потери: {'ner': 25.811911119920747}
Итерация 12/30 — потери: {'ner': 19.184258273430633}
Итерация 13/30 — потери: {'ner': 22.673589035614263}
Итерация 14/30 — потери: {'ner': 25.47910253400237}
Итерация 15/30 — потери: {'ner': 12.678693601230767}
Итерация 16/30 — потери: {'ner': 11.500619597223858}
Итерация 17/30 — потери: {'ner': 13.022053573777983}
Итерация 18/30 — потери: {'ner': 11.314752302103704}
Итерация 19/30 — потери: {'ner': 8.434762258929792}
Итерация 20/3

In [9]:
import spacy
from sklearn.metrics import accuracy_score, f1_score

nlp = spacy.load("ner_gender_model")

true_labels = []
pred_labels = []

def get_full_text(row):
    return f"{str(row['preamble'])} {str(row['description'])} {str(row['sentence'])}"

for idx, row in df_marking.iterrows():
    true_gender = row["gender_accused"]
    if pd.isnull(true_gender) or true_gender == "неизвестно":
        continue 

    text = get_full_text(row)
    doc = nlp(text)

    predicted_gender = "неизвестно"
    for ent in doc.ents:
        if ent.label_ == "GENDER_ACC":
            word = ent.text.lower()
            # print(word)
            if word in female_keywords:
                predicted_gender = "женщина"
            elif word in male_keywords:
                predicted_gender = "мужчина"
            else:
                print(word)
            break

    true_labels.append(true_gender)
    pred_labels.append(predicted_gender)

accuracy = accuracy_score(true_labels, pred_labels)
f1 = f1_score(true_labels, pred_labels, average='weighted') 
print(f"Accuracy: {accuracy:.4f}")
print(f"F1 Score: {f1:.3f}")


Accuracy: 0.9500
F1 Score: 0.950


In [10]:
from sklearn.model_selection import train_test_split
import spacy
from spacy.training.example import Example
from spacy.util import minibatch
import random

train_examples, valid_examples = train_test_split(train_data, test_size=0.2, random_state=42)

nlp = spacy.blank("ru")
ner = nlp.add_pipe("ner")
ner.add_label("GENDER_ACC")

examples = []
for text, annotations in train_examples:
    doc = nlp.make_doc(text)
    examples.append(Example.from_dict(doc, annotations))

n_iter = 20
optimizer = nlp.begin_training()

for i in range(n_iter):
    random.shuffle(examples)
    losses = {}
    batches = minibatch(examples, size=4)

    for batch in batches:
        nlp.update(batch, losses=losses)

    print(f"Итерация {i+1}/{n_iter} — потери: {losses}")

nlp.to_disk("ner_s_gender_model")
print("Модель сохранена в папке 'ner_s_gender_model'")

Итерация 1/20 — потери: {'ner': 123888.09113060695}
Итерация 2/20 — потери: {'ner': 2499.4610717573237}
Итерация 3/20 — потери: {'ner': 119.4512190370946}
Итерация 4/20 — потери: {'ner': 104.84555016882153}
Итерация 5/20 — потери: {'ner': 69.45461433824185}
Итерация 6/20 — потери: {'ner': 55.30262299300204}
Итерация 7/20 — потери: {'ner': 35.735392528839064}
Итерация 8/20 — потери: {'ner': 35.300648705201525}
Итерация 9/20 — потери: {'ner': 27.265650920052437}
Итерация 10/20 — потери: {'ner': 20.543378620329214}
Итерация 11/20 — потери: {'ner': 30.031627662966255}
Итерация 12/20 — потери: {'ner': 15.660521870745367}
Итерация 13/20 — потери: {'ner': 25.662477488433243}
Итерация 14/20 — потери: {'ner': 17.0346381332135}
Итерация 15/20 — потери: {'ner': 16.253721713298177}
Итерация 16/20 — потери: {'ner': 11.215125091724719}
Итерация 17/20 — потери: {'ner': 23.010797015553493}
Итерация 18/20 — потери: {'ner': 10.770997236647531}
Итерация 19/20 — потери: {'ner': 13.90147887242723}
Итерация

In [57]:
from sklearn.metrics import accuracy_score, f1_score

nlp = spacy.load("ner_s_gender_model")

true_labels = []
pred_labels = []

for text, ann in valid_examples:
    doc = nlp(text)

    predicted_gender = "мужчина"
    for ent in doc.ents:
        if ent.label_ == "GENDER_ACC":
            word = ent.text.lower().strip()
            w1 = word.lower()
            if word in male_keywords:
                predicted_gender = "мужчина"
            elif word in female_keywords:
                predicted_gender = "женщина"
            break

    true_gender = ann.get("gender", "неизвестно")
    w2 = text[ann["entities"][0][0]:ann["entities"][0][1]] if ann["entities"] else ""
    w2 = w2

    true_labels.append(true_gender)
    pred_labels.append(predicted_gender)

    if true_gender != predicted_gender:
        print(f"{w1} (предсказано) != {w2} (истинно), true_gender: {true_gender}, predicted: {predicted_gender}")

accuracy = accuracy_score(true_labels, pred_labels)
f1 = f1_score(true_labels, pred_labels, average='weighted') 
print(f"\nAccuracy: {accuracy:.4f}")
print(f"F1 Score: {f1:.3f}")


Accuracy: 1.0000
F1 Score: 1.000
